In [1]:
import pandas as pd
import numpy as np
import os

#import from csv and create win cols
pd.set_option('display.max_rows', 200)
X = pd.read_csv('nfl_team_stats_2002-2024.csv')
X['home'] = X['home'].astype(str)
X['away'] = X['away'].astype(str)
X['home_win'] = X['score_home'] > X['score_away']
X['home_win'] = X['home_win'].astype(int)

X['points_allowed_home'] = X['score_away']
X['points_allowed_away'] = X['score_home']
X['yards_allowed_home'] = X['yards_away']
X['yards_allowed_away'] = X['yards_home']
#if you tie at home means the other team "won" as they dont have home field advantage
X


,season,week,date,time_et,neutral,away,home,score_away,score_home,first_downs_away,...,interceptions_home,def_st_td_away,def_st_td_home,possession_away,possession_home,home_win,points_allowed_home,points_allowed_away,yards_allowed_home,yards_allowed_away
0,2002,1,2002-09-05,8:30 PM,False,49ers,Giants,16,13,13,...,3,0,0,27:32,32:28,0,16,13,279,361
1,2002,1,2002-09-08,1:00 PM,False,Colts,Jaguars,28,25,18,...,1,2,0,27:27,32:33,0,28,25,307,343
2,2002,1,2002-09-08,1:00 PM,False,Cardinals,Commanders,23,31,14,...,1,0,0,25:36,34:24,1,23,31,257,442
3,2002,1,2002-09-08,1:00 PM,False,Lions,Dolphins,21,49,15,...,0,0,2,25:36,34:24,1,21,49,257,389
4,2002,1,2002-09-08,1:00 PM,False,Eagles,Titans,24,27,17,...,1,0,0,29:12,30:48,1,24,27,261,328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6209,2024,Division,2025-01-19,3:00 PM,False,Rams,Eagles,22,28,18,...,0,0,0,29:06,30:54,1,22,28,402,350
6210,2024,Division,2025-01-19,6:30 PM,False,Ravens,Bills,25,27,23,...,0,0,0,28:16,31:44,1,25,27,416,273
6211,2024,Conference,2025-01-26,3:00 PM,False,Commanders,Eagles,23,55,22,...,0,0,0,29:29,30:31,1,23,55,350,459
6212,2024,Conference,2025-01-26,6:30 PM,False,Bills,Chiefs,29,32,22,...,0,0,0,30:32,29:28,1,29,32,374,368


In [3]:
def poss_to_seconds(val):
    try:
        m, s = map(int, val.split(':'))
        return m * 60 + s
    except:
        return np.nan
    
#change pos to singular number 
X.loc[:, 'possession_home'] = X['possession_home'].apply(poss_to_seconds)
X.loc[:, 'possession_away'] = X['possession_away'].apply(poss_to_seconds)
X.loc[:,'week'] = pd.to_numeric(X['week'], errors='coerce')
X = X.dropna(subset=['week'])
X= X.copy()


X.loc[:,'game_id'] = X.index



In [4]:
#Find Home Cols and Away cols and split up dataframe based on them to simplify cols and be able to track each team whether they are home or away
home_cols = [col for col in X.columns if col.endswith('_home')]
away_cols = [col for col in X.columns if col.endswith('_away')]

home_df = X[['season', 'week', 'game_id', 'home', 'home_win'] + home_cols].copy()
away_df = X[['season', 'week', 'game_id', 'away'] + away_cols].copy()

home_df = home_df.rename(columns=lambda x: x.replace('_home', '') if x.endswith('_home') else x)
away_df = away_df.rename(columns=lambda x: x.replace('_away', '') if x.endswith('_away') else x)

home_df = home_df.rename(columns={'home': 'team'})
away_df = away_df.rename(columns={'away': 'team'})

In [5]:
long_X = pd.concat([home_df, away_df], ignore_index=True)
long_X = long_X.sort_values(by =['season', 'team', 'week'])
#print(long_X.head(200))

stat_cols = [col for col in long_X.columns if col not in ['season', 'week', 'team', 'game_id', 'home_win']]
print(stat_cols)

#rolling average, shifted by one so all we know are the stats leading up to that game for the current season averages.
rolling_avg = (
    long_X
    .groupby(['season', 'team'])[stat_cols]
    .transform(lambda x: x.shift(1).expanding().mean())
)
print(rolling_avg)

long_X = long_X.sort_values(by=['season', 'team', 'week'])
rolling_X = pd.concat([long_X[['season', 'week', 'team', 'game_id', 'home_win']], rolling_avg], axis=1)
print(rolling_X.head(30))

#merge the two df back together and label to be able to tell which team was the home team
home_stats = rolling_X.merge(X[['game_id', 'home']], left_on=['game_id', 'team'], right_on=['game_id', 'home'])
away_stats = rolling_X.merge(X[['game_id', 'away']], left_on=['game_id', 'team'], right_on=['game_id', 'away'])

home_stats = home_stats.add_suffix('_home')
away_stats = away_stats.add_suffix('_away')

#final merge of home and away stats
final_X = pd.merge(home_stats, away_stats, left_on='game_id_home', right_on='game_id_away')
final_X = final_X.rename(columns={'game_id_home': 'game_id'}).drop(columns=['game_id_away'])

#drop cols that are week 1 as we do not have any existing information about their season yet.
final_X = final_X[final_X['week_home'] > 1].copy()


['score', 'first_downs', 'first_downs_from_passing', 'first_downs_from_rushing', 'first_downs_from_penalty', 'third_down_comp', 'third_down_att', 'fourth_down_comp', 'fourth_down_att', 'plays', 'drives', 'yards', 'pass_comp', 'pass_att', 'pass_yards', 'sacks_num', 'sacks_yards', 'rush_att', 'rush_yards', 'pen_num', 'pen_yards', 'redzone_comp', 'redzone_att', 'fumbles', 'interceptions', 'def_st_td', 'possession', 'points_allowed', 'yards_allowed']
           score  first_downs  first_downs_from_passing  \
5951         NaN          NaN                       NaN   
29     16.000000    13.000000                  7.000000   
41     15.000000    15.500000                  9.000000   
70     16.666667    17.666667                  8.333333   
6038   21.750000    18.500000                  8.750000   
...          ...          ...                       ...   
5876   24.750000    20.583333                 11.833333   
5901   26.076923    20.769231                 12.076923   
11864  26.357143  

In [6]:
#tweak the names for more readability and drop repeat cols that we do not need.
final_X['week'] = final_X['week_home']
final_X['week'] = final_X['week_home']
final_X['season'] = final_X['season_home']
final_X['home_win'] = final_X['home_win_home']
final_X = final_X.drop(['season_home', 'week_home', 'home_home', 'season_away', 'week_away', 'away_away', 'home_win_home', 'home_win_away'], axis =1)

print(final_X.columns)
final_X.head(30)

Index(['team_home', 'game_id', 'score_home', 'first_downs_home',
       'first_downs_from_passing_home', 'first_downs_from_rushing_home',
       'first_downs_from_penalty_home', 'third_down_comp_home',
       'third_down_att_home', 'fourth_down_comp_home', 'fourth_down_att_home',
       'plays_home', 'drives_home', 'yards_home', 'pass_comp_home',
       'pass_att_home', 'pass_yards_home', 'sacks_num_home',
       'sacks_yards_home', 'rush_att_home', 'rush_yards_home', 'pen_num_home',
       'pen_yards_home', 'redzone_comp_home', 'redzone_att_home',
       'fumbles_home', 'interceptions_home', 'def_st_td_home',
       'possession_home', 'points_allowed_home', 'yards_allowed_home',
       'team_away', 'score_away', 'first_downs_away',
       'first_downs_from_passing_away', 'first_downs_from_rushing_away',
       'first_downs_from_penalty_away', 'third_down_comp_away',
       'third_down_att_away', 'fourth_down_comp_away', 'fourth_down_att_away',
       'plays_away', 'drives_away', 'yard

,team_home,game_id,score_home,first_downs_home,first_downs_from_passing_home,first_downs_from_rushing_home,first_downs_from_penalty_home,third_down_comp_home,third_down_att_home,fourth_down_comp_home,...,redzone_att_away,fumbles_away,interceptions_away,def_st_td_away,possession_away,points_allowed_away,yards_allowed_away,week,season,home_win
0,49ers,29,16.000000,13.000000,7.000000,5.000000,1.000000,4.000000,12.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,NaN,16.000000,327.000000,2.0,2002,0.0
1,49ers,41,15.000000,15.500000,9.000000,6.000000,0.500000,6.000000,12.500000,0.000000,...,5.500000,0.500000,1.500000,0.500000,NaN,30.000000,354.000000,3.0,2002,1.0
2,49ers,70,16.666667,17.666667,8.333333,9.000000,0.333333,6.000000,13.000000,0.000000,...,4.250000,1.000000,2.250000,0.000000,NaN,22.000000,313.000000,5.0,2002,1.0
3,49ers,111,23.666667,20.666667,10.666667,9.000000,1.000000,6.333333,12.000000,0.000000,...,3.666667,0.500000,0.833333,0.500000,NaN,15.500000,334.333333,8.0,2002,1.0
4,49ers,141,25.375000,21.500000,10.875000,9.250000,1.375000,7.125000,13.000000,0.250000,...,7.875000,0.125000,1.250000,0.000000,NaN,30.000000,438.000000,10.0,2002,1.0
5,49ers,175,23.700000,21.500000,11.200000,9.000000,1.300000,7.600000,14.000000,0.400000,...,6.000000,0.900000,0.600000,0.000000,NaN,16.700000,282.900000,12.0,2002,0.0
6,49ers,189,23.090909,21.727273,11.727273,8.727273,1.272727,7.727273,14.545455,0.454545,...,6.090909,0.545455,0.818182,0.454545,NaN,22.636364,379.909091,13.0,2002,1.0
7,49ers,221,24.307692,22.000000,11.923077,8.846154,1.230769,8.000000,15.076923,0.538462,...,6.461538,0.923077,1.000000,0.615385,NaN,20.923077,323.076923,15.0,2002,0.0
9,Bears,35,20.500000,16.500000,10.000000,5.500000,1.000000,5.500000,13.000000,0.000000,...,4.000000,0.000000,1.500000,2.000000,NaN,20.000000,345.000000,3.0,2002,0.0
10,Bears,73,22.750000,17.250000,11.000000,4.500000,1.750000,5.750000,13.750000,0.250000,...,5.500000,1.500000,0.750000,0.500000,NaN,28.500000,333.500000,5.0,2002,0.0


In [ ]:
#lets convert this into a csv file so myself and others can use in in ML models
final_X.to_csv('final_nfl_rolling_stats.csv', index=False)